[![cloudevel](../img/cloudevel.png)](https://cloudevel.com)

# Solución: workflow que utiliza expresiones

**Cómo leer este cuaderno.** Cada ejercicio reproduce el enunciado del cuaderno original y presenta su solución.

* Las celdas **sin etiqueta** se ejecutan en cualquier equipo, sin credenciales ni cuenta de *GitHub*. Escriben solo en carpetas temporales.
* Las celdas con la etiqueta `skip-execution` necesitan una cuenta de *GitHub*, un proyecto de nube o un servidor; la integración continua (*CI*) no las ejecuta. Debajo de cada una se describe la salida esperada.
* Ningún valor real aparece en el código: los datos de cada persona se leen de variables de entorno o se sustituyen por los marcadores indicados.

Este cuaderno resuelve `07_actividad_expresiones.ipynb`. Parte de las dos entidades del cuaderno 06: la variable `VARIABLE_DEMO` y el secreto `MUY_SECRETO`.

La siguiente celda importará el módulo `utilidades` (que está en esta misma carpeta) y localizará la raíz del repositorio. Todas las comprobaciones locales de este cuaderno leen archivos desde esa raíz.

In [ ]:
import utilidades as u

RAIZ = u.raiz_repositorio()
RAIZ.name

## Ejercicio 1. Crear el workflow `expresiones.yml`

En la carpeta `.github/workflows` del repositorio de la actividad previa, crear un archivo nuevo con el *workflow* `Expresiones`, que imprime una variable local, una variable del repositorio y un secreto. El enunciado original pide «copiar y pegar» el código y no indica el nombre del archivo; el registro de la ejecución muestra `Create expresiones.yml`, de modo que el nombre es `expresiones.yml`.

La siguiente celda definirá `EXPRESIONES`, el texto del *workflow* del cuaderno original. Tiene un disparador manual, una variable `VARIABLE_LOCAL` en el nivel del archivo y tres pasos `run` que usan los contextos `env`, `vars` y `secrets`.

In [ ]:
EXPRESIONES = """\
name: Expresiones

env:
  VARIABLE_LOCAL: soy una variable local.

on:
  workflow_dispatch:

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@3d3c42e5aac5ba805825da76410c181273ba90b1 # v7.0.1

      - name: Despliega variable local
        run: echo $VARIABLE_LOCAL

      - name: Despliega variable del repositorio
        run: echo ${{ vars.VARIABLE_DEMO }}

      - name: No despliega secreto del repositorio
        run: echo ${{ secrets.MUY_SECRETO }}
"""

La siguiente celda comprobará la estructura del *workflow*: un solo disparador manual, un *job* `build` con cuatro pasos y las acciones fijadas por *SHA*.

In [ ]:
wf = u.cargar_workflow(EXPRESIONES)
assert wf["name"] == "Expresiones"
assert list(wf["on"]) == ["workflow_dispatch"]
assert wf["env"] == {"VARIABLE_LOCAL": "soy una variable local."}
assert len(u.pasos(wf, "build")) == 4
assert u.acciones_sin_fijar(wf) == []
print("Estructura correcta")

## Ejercicio 2. Guardar el archivo y ejecutar el workflow

Guardar el archivo con un *commit* en la rama principal, abrir el *workflow* `Expresiones` en el menú lateral de `Actions`, ejecutarlo y abrir el *job* `build`.

La siguiente celda ensayará el *workflow* en una carpeta temporal. Los contextos de ejemplo hacen de `vars` y `secrets`. La función de ensayo enmascara el secreto con `***`, como hace *GitHub* en sus registros.

In [ ]:
SECRETO = "valor-de-demostracion"
contextos = {
    "vars": {"VARIABLE_DEMO": "Hola, Mundo."},
    "secrets": {"MUY_SECRETO": SECRETO},
}
resultados = u.simular_job(wf, "build", contextos, u.directorio_temporal(), secretos=(SECRETO,))
u.mostrar_resultados(resultados)

Las tres líneas de salida son las que describe la actividad: la variable local, el valor de la variable del repositorio y el secreto enmascarado.

La siguiente celda verificará esas tres salidas con `assert`. El secreto jamás debe aparecer en claro.

In [ ]:
salidas = {nombre: salida for nombre, salida, _ in resultados}
assert salidas["Despliega variable local"] == "soy una variable local."
assert salidas["Despliega variable del repositorio"] == "Hola, Mundo."
assert salidas["No despliega secreto del repositorio"] == "***"
assert SECRETO not in "".join(salida for _, salida, _ in resultados)
print("Salidas verificadas")

La siguiente celda enviará el archivo al repositorio, ejecutará el *workflow* y mostrará el registro del *job*. Requiere haber creado `VARIABLE_DEMO` y `MUY_SECRETO` (cuaderno 06).

In [ ]:
import subprocess
from pathlib import Path

clon = Path.home() / "demo-github-actions"
destino = clon / ".github" / "workflows" / "expresiones.yml"
destino.write_text(EXPRESIONES, encoding="utf-8")
for comando in (
    ["git", "-C", str(clon), "add", ".github/workflows/expresiones.yml"],
    ["git", "-C", str(clon), "commit", "--message", "Create expresiones.yml"],
    ["git", "-C", str(clon), "push", "origin", "main"],
    ["gh", "workflow", "run", "expresiones.yml", "--ref", "main"],
):
    subprocess.run(comando, check=True, cwd=clon)

La siguiente celda esperará el fin de la ejecución y mostrará solo las líneas de salida de los tres pasos.

In [ ]:
%%bash
set -euo pipefail
cd "${HOME}/demo-github-actions"
sleep 5
ID="$(gh run list --workflow expresiones.yml --limit 1 --json databaseId --jq '.[0].databaseId')"
gh run watch "${ID}" --exit-status
gh run view "${ID}" --log | grep -E "Despliega|Hola|variable local|\*\*\*"

**Salida esperada.** El registro contiene `soy una variable local.`, `Hola, Mundo.` y `***` en lugar del valor del secreto. El mensaje de *GitHub* para este último puede mostrar el comando como `echo ***`.

## Ejercicio 3. Observación de seguridad (complemento)

El paso `echo ${{ secrets.MUY_SECRETO }}` inserta el valor del secreto dentro del texto del guion antes de ejecutarlo. Indicar por qué eso es una mala práctica y cuál es la alternativa.

*Este ejercicio no está en el cuaderno original; enlaza esta actividad con el cuaderno 13.*

*GitHub* sustituye la expresión por el valor **antes** de que el *shell* lea el guion. Un valor con comillas, punto y coma o `$(…)` cambiaría el programa que se ejecuta (inyección de comandos). La alternativa es pasar el valor por una variable de entorno del paso y leerla con comillas: el *shell* la trata como dato, no como código.

La siguiente celda comparará las dos formas con un valor malicioso de ejemplo. En la forma insegura, el texto `; echo INYECTADO` se ejecuta como un comando aparte. En la segura, se imprime como texto.

In [ ]:
import subprocess

VALOR = "ok; echo INYECTADO"

inseguro = u.sustituir_expresiones("echo ${{ secrets.X }}", {"secrets": {"X": VALOR}})
seguro = 'echo "${X}"'

salida_insegura = subprocess.run(["bash", "-c", inseguro], capture_output=True, text=True, check=True).stdout
salida_segura = subprocess.run(["bash", "-c", seguro], env={"X": VALOR, "PATH": "/usr/bin:/bin"},
                               capture_output=True, text=True, check=True).stdout
print("Insegura:", salida_insegura.splitlines())
print("Segura:  ", salida_segura.splitlines())

assert salida_insegura.splitlines() == ["ok", "INYECTADO"]
assert salida_segura.splitlines() == ["ok; echo INYECTADO"]

La versión segura del paso del *workflow* sería:

```yaml
- name: Usa el secreto como dato
  env:
    MUY_SECRETO: ${{ secrets.MUY_SECRETO }}
  run: test -n "${MUY_SECRETO}" && echo "El secreto está definido"
```

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2027.</p>